In [1]:
import sys; sys.path.insert(0, "..")  # notebook is in notebooks/, pipeline/ is one level up
from collections import Counter
import numpy as np
from pipeline.embeddings import download_pretrained, PretrainedEmbeddings
from pipeline.tokenization import BasicTokenizer
from pipeline.data import load_saved_folds

path, dims, skip = download_pretrained("glove")  # file is already there, so no download
emb = PretrainedEmbeddings(path, dims, skip_first_line=skip)
tok = BasicTokenizer(lowercase=True, remove_punct=False, remove_stopwords=False, vocab=emb.vocab)  # exactly the model's tokenizer
subset, folds = load_saved_folds("../results/data/mnli-n50000-seed0")
print(len(emb.vocab), "GloVe words |", len(subset), "MNLI examples")

1287615 GloVe words | 50001 MNLI examples


In [2]:
def coverage(texts):
    tokens = [t for text in texts for t in tok.tokens(text)]
    counts = Counter(tokens)
    oov = Counter({w: c for w, c in counts.items() if w not in emb})  # OOV = no GloVe vector -> becomes <unk>
    stats = {
        "tokens": len(tokens),
        "token_coverage": 1 - sum(oov.values()) / len(tokens),  # share of all word occurrences
        "types": len(counts),
        "type_coverage": 1 - len(oov) / len(counts),            # share of distinct words
    }
    return stats, oov

for col in ["premise", "hypothesis"]:
    stats, oov = coverage(subset[col])
    print(col, {k: round(v, 4) if isinstance(v, float) else v for k, v in stats.items()})
    print("  most frequent OOV:", oov.most_common(20))

premise {'tokens': 1115631, 'token_coverage': 0.99, 'types': 26666, 'type_coverage': 0.9}
  most frequent OOV: [('um-hum', 279), ("ca'daan", 192), ('inglethorp', 105), ('long-term', 75), ('17th-century', 57), ('low-income', 57), ('vandemeyer', 54), ("san'doro", 51), ('bye-bye', 48), ('h-2a', 45), ('follow-up', 33), ('16th-century', 33), ('18th-century', 33), ('old-fashioned', 30), ('first-class', 30), ('14th-century', 30), ('19th-century', 27), ('open-air', 27), ('one-third', 24), ('hersheimmer', 24)]
hypothesis {'tokens': 562672, 'token_coverage': 0.9945, 'types': 23607, 'type_coverage': 0.931}
  most frequent OOV: [("ca'daan", 139), ('inglethorp', 72), ('vandemeyer', 43), ("san'doro", 41), ('..', 38), ('long-term', 23), ('h-2a', 21), ('low-income', 15), ('hersheimmer', 15), ('bauerstein', 15), ('open-air', 13), ('al-ghuri', 10), ('willingness-to-pay', 9), ('non-households', 9), ('first-class', 9), ('satheri', 9), ('well-known', 9), ('short-term', 9), ('tubacca', 9), ('gauve', 9)]


In [3]:
def cos(a, b):
    assert a in emb and b in emb, f"{a}/{b} not in GloVe -> would compare <unk>"
    va, vb = emb.embed_word(a), emb.embed_word(b)
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

probes = {
    "synonym":   [("big", "large"), ("happy", "glad"), ("car", "automobile"), ("begin", "start")],
    "antonym":   [("good", "bad"), ("happy", "sad"), ("big", "small"), ("always", "never"), ("win", "lose")],
    "hypernym":  [("dog", "animal"), ("car", "vehicle"), ("rose", "flower"), ("hammer", "tool")],
    "unrelated": [("dog", "government"), ("car", "happy"), ("flower", "tax")],
}
for kind, pairs in probes.items():
    sims = [cos(a, b) for a, b in pairs]
    print(f"{kind:9s} mean={np.mean(sims):.2f}", [(a, b, round(s, 2)) for (a, b), s in zip(pairs, sims)])

synonym   mean=0.81 [('big', 'large', 0.74), ('happy', 'glad', 0.88), ('car', 'automobile', 0.72), ('begin', 'start', 0.9)]
antonym   mean=0.80 [('good', 'bad', 0.81), ('happy', 'sad', 0.74), ('big', 'small', 0.76), ('always', 'never', 0.92), ('win', 'lose', 0.78)]
hypernym  mean=0.65 [('dog', 'animal', 0.78), ('car', 'vehicle', 0.9), ('rose', 'flower', 0.45), ('hammer', 'tool', 0.45)]
unrelated mean=0.34 [('dog', 'government', 0.31), ('car', 'happy', 0.53), ('flower', 'tax', 0.17)]


In [4]:
norm = emb.vectors / (np.linalg.norm(emb.vectors, axis=1, keepdims=True) + 1e-8)  # unit length -> dot product = cosine

def neighbours(word, k=8):
    sims = norm @ norm[emb.index(word)]       # cosine with all 1.2M words at once
    best = np.argsort(-sims)[1:k + 1]         # [1:] skips the word itself
    return [(emb.word(i), round(float(sims[i]), 2)) for i in best]

for w in ["not", "never", "no", "nobody", "sleeping"]:
    print(w, neighbours(w))

not [('be', 0.97), ('if', 0.97), ('because', 0.96), ('should', 0.96), ('could', 0.96), ('any', 0.95), ('but', 0.95), ('whether', 0.95)]
never [('did', 0.94), ('so', 0.93), ('actually', 0.93), ('thought', 0.92), ('but', 0.92), ('not', 0.92), ('why', 0.92), ('what', 0.92)]
no [('not', 0.94), ('any', 0.93), ('nothing', 0.92), ('if', 0.91), ('because', 0.91), ('what', 0.91), ('...', 0.91), ('without', 0.9)]
nobody [('else', 0.96), ('anybody', 0.95), ('everybody', 0.95), ('somebody', 0.94), ('everyone', 0.94), ('know', 0.93), ('me', 0.92), ('anyone', 0.92)]
sleeping [('bed', 0.89), ('sleep', 0.82), ('slept', 0.82), ('asleep', 0.8), ('bathroom', 0.8), ('walking', 0.77), ('room', 0.77), ('sick', 0.76)]
